# Week 8 · Day 5 (companion) — The Same Real Network in TensorFlow / Keras

You just trained a real MLP on the handwritten digits in **PyTorch** — the full workflow: load, split, scale, build, train, evaluate, and turn the practical knobs. This companion does the **exact same job in TensorFlow / Keras**, so you can read a real-dataset training script in either framework.

Same dataset, same architecture, same four moves. As on Day 4: only the words change. The big difference you'll feel again — **Keras writes the training loop for you** with `.fit()`, where PyTorch had you write it out.

**The plan (mirrors the PyTorch notebook):**
1. Load and prepare the digits (identical prep).
2. Build the MLP — Keras style.
3. Train with `.fit()` — the loop, handled for you.
4. Evaluate — accuracy + confusion matrix.
5. Turn the same practical knobs.
6. Read the PyTorch ↔ Keras cheat sheet.

In [ ]:
# install if needed (Colab already has these)
# !pip install tensorflow scikit-learn

import tensorflow as tf
from tensorflow import keras
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

tf.random.set_seed(42)
np.random.seed(42)
print("TensorFlow version:", tf.__version__)

---
## 1. Load and prepare the data

This part is **identical** to the PyTorch notebook — the data prep has nothing to do with the framework. Same digits, same split, same scaling. The only thing we *don't* do here is convert to tensors: Keras takes plain NumPy arrays directly.

In [ ]:
digits = load_digits()
X = digits.data       # (1797, 64)
y = digits.target     # (1797,)

# split — 80/20, stratified
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

# scale — fit on train only
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print("train:", X_train.shape, " test:", X_test.shape)
print("classes:", np.unique(y))

One small difference from PyTorch worth noting: in PyTorch we made the labels `long` tensors and used `CrossEntropyLoss`. In Keras we keep labels as plain integers and use **`sparse_categorical_crossentropy`** — same loss, the "sparse" name just means "labels are integers, not one-hot." Keras handles the rest.

---
## 2. Build the network

Same architecture as the PyTorch version — **64 → 64 → 32 → 10**, ReLU on the hidden layers, raw scores out. Compare the two side by side:

| PyTorch | Keras |
|---|---|
| `nn.Linear(64, 64)` | `Dense(64)` (after `Input(shape=(64,))`) |
| `nn.ReLU()` | `activation="relu"` |
| `nn.Linear(32, 10)` | `Dense(10)` |

And just like PyTorch, **no softmax on the final layer** — we output 10 raw scores and let the loss handle it (via `from_logits=True` at compile time).

In [ ]:
tf.random.set_seed(42)

model = keras.Sequential([
    keras.layers.Input(shape=(64,)),                 # 64 pixels in
    keras.layers.Dense(64, activation="relu"),       # 64 -> 64 hidden
    keras.layers.Dense(32, activation="relu"),       # 64 -> 32 hidden
    keras.layers.Dense(10)                           # 32 -> 10 outputs (raw scores)
])

model.summary()

### Compile — pick loss and optimizer
In PyTorch these were two separate objects (`loss_fn` and `optimizer`). Keras bundles them into one `compile` call — same two choices:
- **Adam** optimizer, learning rate 0.01 (identical to PyTorch)
- **sparse categorical crossentropy** loss, with `from_logits=True` because our last layer outputs raw scores (no softmax)

In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.01),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"]
)
print("compiled — loss + optimizer chosen, same as PyTorch")

---
## 3. Train with `.fit()`

Here's the Day 4 lesson again, now on real data. In PyTorch you wrote the whole loop: iterate batches, forward, loss, `zero_grad`, `backward`, `step`. Keras does all of that inside **one `.fit()` call** — you just hand it the data, the batch size, and the number of epochs.

The four moves are still happening once per batch, per epoch — Keras just writes the loop. Because we pass `validation_data`, it also reports **test loss/accuracy every epoch**, so we can watch for overfitting exactly as before.

In [ ]:
history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=50,
    batch_size=32,
    verbose=0                     # quiet; set to 1 to watch it train live
)

print(f"final train loss: {history.history['loss'][-1]:.4f}")
print(f"final test  loss: {history.history['val_loss'][-1]:.4f}")

That single `.fit()` replaced the entire PyTorch training loop — the `DataLoader`, the batch iteration, the four in-loop lines, and the per-epoch bookkeeping. That's the Keras trade-off: less code, but the loop is hidden. You know what's inside because you wrote it by hand this week.

In [ ]:
plt.plot(history.history["loss"], label="train loss", color="purple")
plt.plot(history.history["val_loss"], label="test loss", color="orange")
plt.xlabel("epoch"); plt.ylabel("loss")
plt.title("Training and test loss (Keras)")
plt.legend(); plt.grid(alpha=0.3)
plt.show()

Same read as the PyTorch plot: as long as test loss falls with train loss, the network is genuinely learning. A rising test loss while train keeps falling would be overfitting.

---
## 4. Evaluate — the familiar toolkit

Accuracy on the held-out test set, then a confusion matrix — same tools, same read as the SVM and the PyTorch network.

In [ ]:
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f"test accuracy: {test_acc:.2%}")

# predictions: model outputs 10 raw scores per image; argmax picks the digit
logits = model.predict(X_test, verbose=0)
predictions = logits.argmax(axis=1)

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

cm = confusion_matrix(y_test, predictions)
fig, ax = plt.subplots(figsize=(7, 6))
ConfusionMatrixDisplay(cm, display_labels=range(10)).plot(ax=ax, cmap="Blues", colorbar=False)
plt.title(f"Confusion matrix — {test_acc:.1%} accuracy (Keras)")
plt.show()

### Look at the mistakes
Same useful habit — see which digits fooled the network.

In [ ]:
wrong = np.where(predictions != y_test)[0]
print(f"the network got {len(wrong)} of {len(y_test)} test images wrong")

if len(wrong) > 0:
    show = wrong[:8]
    fig, axes = plt.subplots(1, len(show), figsize=(2*len(show), 2.5))
    axes = np.atleast_1d(axes)
    X_test_orig = scaler.inverse_transform(X_test)   # undo scaling to see the image
    for ax, idx in zip(axes, show):
        ax.imshow(X_test_orig[idx].reshape(8, 8), cmap="gray")
        ax.set_title(f"pred {predictions[idx]}\ntrue {y_test[idx]}", fontsize=9)
        ax.axis("off")
    plt.suptitle("Where the network slipped up")
    plt.tight_layout()
    plt.show()

---
## 5. The practical knobs

The same three knobs from the PyTorch notebook — learning rate, network size, epochs — behave the same way here, because they're properties of *neural networks*, not of any one framework. We wrap training in a small function and turn one knob at a time.

(Notice how compact the Keras training function is — the whole train step is one `.fit()`.)

In [ ]:
def train_and_score(hidden=(64, 32), lr=0.01, epochs=50, seed=42):
    """Build + train a Keras MLP with the given knobs; return train/test accuracy."""
    tf.random.set_seed(seed)
    layers = [keras.layers.Input(shape=(64,))]
    for h in hidden:
        layers.append(keras.layers.Dense(h, activation="relu"))
    layers.append(keras.layers.Dense(10))
    net = keras.Sequential(layers)

    net.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr),
        loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
        metrics=["accuracy"])
    net.fit(X_train, y_train, epochs=epochs, batch_size=32, verbose=0)

    tr = net.evaluate(X_train, y_train, verbose=0)[1]
    te = net.evaluate(X_test, y_test, verbose=0)[1]
    return tr, te

### Knob 1 — learning rate
The most important setting. Too small and it underfits; too big and training destabilizes.

In [ ]:
print("learning rate   train acc   test acc")
for lr in [0.0001, 0.001, 0.01, 0.1, 1.0]:
    tr, te = train_and_score(lr=lr, epochs=30)
    print(f"   {lr:<10}    {tr:.2%}      {te:.2%}")
print("\nToo small → underfits. ~0.001-0.01 → strong. Too big → unstable, collapses.")

### Knob 2 — network size
More capacity fits training better, but watch the train-test gap — the tell for memorizing rather than learning.

In [ ]:
print("network size          train acc   test acc   gap")
for hidden in [(16,), (64, 32), (256, 128, 64)]:
    tr, te = train_and_score(hidden=hidden, epochs=40)
    print(f"   {str(hidden):<18}  {tr:.2%}      {te:.2%}    {tr-te:+.2%}")
print("\nAll sizes fit training near-perfectly; the GAP is the tell.")
print("On this small clean dataset it stays modest — on messier data it grows with size.")

### Knob 3 — epochs
Too few underfits; past convergence, extra epochs stop helping (and on messy data, invite overfitting).

In [ ]:
print("epochs   train acc   test acc")
for ep in [5, 20, 50, 150]:
    tr, te = train_and_score(epochs=ep)
    print(f"  {ep:<5}   {tr:.2%}      {te:.2%}")
print("\nFew (5) → underfitting. ~20-50 → converged. 150 → diminishing returns.")

The knobs behave exactly as they did in PyTorch — because they belong to the network, not the framework. Learning rate = *how fast*, size = *how much*, epochs = *how long*.

---
## 6. PyTorch ↔ Keras — the real-dataset cheat sheet

You've now done the full real-data workflow in both. Here's the whole thing mapped side by side:

| Step | PyTorch | Keras |
|---|---|---|
| Prepare data (split/scale) | sklearn — **identical** | sklearn — **identical** |
| To tensors | `torch.tensor(...)` | not needed (NumPy is fine) |
| Build model | `nn.Sequential(nn.Linear, nn.ReLU, ...)` | `keras.Sequential([Dense(activation="relu"), ...])` |
| Multi-class loss | `nn.CrossEntropyLoss()` | `SparseCategoricalCrossentropy(from_logits=True)` |
| Optimizer | `torch.optim.Adam(...)` | `keras.optimizers.Adam(...)` |
| Batches | `DataLoader(..., batch_size=32)` | `batch_size=32` inside `.fit()` |
| **Train** | **you write the loop** | **`model.fit(...)`** |
| Predict | `model(X).argmax(1)` | `model.predict(X).argmax(1)` |
| Evaluate | compute accuracy yourself | `model.evaluate(X, y)` |

**The one real difference, again:** PyTorch makes the training loop visible (you write forward → loss → backward → update); Keras hides it inside `.fit()`. Everything else is renamed, not rethought.

We use **PyTorch as our main framework** for the rest of the course — seeing the loop keeps the mechanics in view while you learn, and it's what most modern deep-learning research is written in. But you can now pick up a Keras notebook and read it without missing a beat.

---
## Summary

- The **same real MLP** trained on the same digits, now in Keras — same architecture, same result.
- Data prep (split, scale) is framework-independent; only the model, loss, and training call change names.
- Keras multi-class essentials: `Dense(10)` output (no softmax) + `SparseCategoricalCrossentropy(from_logits=True)`.
- `.fit()` **is** the PyTorch training loop, written for you — the four moves still run once per batch, per epoch.
- The **practical knobs behave identically** across frameworks, because they belong to the network.
- You can now read and write a real-dataset training script in **either** framework.

> **PyTorch shows you the loop; Keras writes it for you. Same network, same digits, same answer.**

With Week 8 complete, you understand neural networks from a single hand-built neuron all the way to a trained, tuned, real-data classifier — in both major frameworks. **Next week: CNNs**, where networks built for images finally leave the SVM behind.